# TỔNG HỢP SO SÁNH HIỆU SUẤT: LƯỢNG TỬ vs CỔ ĐIỂN
Notebook này nạp đồng thời toàn bộ 4 file báo cáo Excel và trích xuất tinh hoa thành 5 biểu đồ so sánh gọn gàng, tinh tế nhất.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Thiết lập màu sắc
COLOR_QUANTUM = '#1f77b4'  # Xanh lam
COLOR_CLASSIC = '#ffa500'  # Vàng cam

# 1. NẠP 4 FILE DỮ LIỆU
files = {
    'c_99': 'classic - 0.9-0.9.xlsx',
    'q_99': 'quantum - 0.9-0.9.xlsx',
    'c_98': 'classic - 0.9-0.85.xlsx',
    'q_98': 'quantum - 0.9-0.85.xlsx'
}

dfs = {}
for key, path in files.items():
    if not os.path.exists(path):
        print(f"⚠️ Lỗi: Không tìm thấy file {path}")
        continue
    df = pd.read_excel(path)
    df['Model Core'] = df['Model Core'].ffill()
    if 'Model Core' not in df.columns:
        df = df.reset_index()
    dfs[key] = df

if len(dfs) == 4:
    print("✅ Đã nạp thành công cả 4 file dữ liệu!")
    
    # Lấy danh sách mô hình (từ bất kỳ file nào)
    models = [m for m in dfs['c_99']['Model Core'].unique() if pd.notna(m)]
    
    # 2. BÓC TÁCH DỮ LIỆU CHO 5 LOẠI BIỂU ĐỒ
    
    # --- Chart 1 & 2: AUC và Max F1 (Lấy từ file 0.9-0.9, hàng Cân Bằng) ---
    c_f1 = dfs['c_99'][dfs['c_99']['Phiên bản Kinh doanh'].str.contains('Cân Bằng', na=False, case=False)]
    q_f1 = dfs['q_99'][dfs['q_99']['Phiên bản Kinh doanh'].str.contains('Cân Bằng', na=False, case=False)]
    
    # --- Chart 3: Precision khi Recall >= 0.9 (Lấy từ file 0.9-0.9, hàng Quét Sạch) ---
    c_prec_at_rec90 = dfs['c_99'][dfs['c_99']['Phiên bản Kinh doanh'].str.contains('Quét Sạch', na=False, case=False)]
    q_prec_at_rec90 = dfs['q_99'][dfs['q_99']['Phiên bản Kinh doanh'].str.contains('Quét Sạch', na=False, case=False)]
    
    # --- Chart 4: Precision khi Recall >= 0.85 (Lấy từ file 0.9-0.85, hàng Quét Sạch) ---
    c_prec_at_rec85 = dfs['c_98'][dfs['c_98']['Phiên bản Kinh doanh'].str.contains('Quét Sạch', na=False, case=False)]
    q_prec_at_rec85 = dfs['q_98'][dfs['q_98']['Phiên bản Kinh doanh'].str.contains('Quét Sạch', na=False, case=False)]
    
    # --- Chart 5: Recall khi Precision >= 0.9 (Lấy từ file 0.9-0.9, hàng An Toàn) ---
    c_rec_at_prec90 = dfs['c_99'][dfs['c_99']['Phiên bản Kinh doanh'].str.contains('An Toàn', na=False, case=False)]
    q_rec_at_prec90 = dfs['q_99'][dfs['q_99']['Phiên bản Kinh doanh'].str.contains('An Toàn', na=False, case=False)]

    # 3. VẼ BIỂU ĐỒ GỘP (Layout: 2 hàng. Hàng trên 3 hình, hàng dưới 2 hình căn giữa)
    fig = plt.figure(figsize=(22, 14))
    fig.suptitle("ĐÁNH GIÁ TOÀN DIỆN: QUANTUM vs CLASSIC TRÊN 7 MÔ HÌNH SOTA", fontsize=24, fontweight='bold', y=1.05)
    
    # Khởi tạo các khung vẽ (GridSpec)
    gs = fig.add_gridspec(2, 6)
    ax1 = fig.add_subplot(gs[0, 0:2]) # Row 0, Left
    ax2 = fig.add_subplot(gs[0, 2:4]) # Row 0, Middle
    ax3 = fig.add_subplot(gs[0, 4:6]) # Row 0, Right
    ax4 = fig.add_subplot(gs[1, 1:3]) # Row 1, Center-Left
    ax5 = fig.add_subplot(gs[1, 3:5]) # Row 1, Center-Right
    
    def plot_line(ax, df_c, df_q, metric, title):
        m_c = df_c.drop_duplicates(subset=['Model Core'], keep='last').set_index('Model Core').reindex(models)[metric]
        m_q = df_q.drop_duplicates(subset=['Model Core'], keep='last').set_index('Model Core').reindex(models)[metric]
        
        ax.plot(models, m_c, marker='s', markersize=9, color=COLOR_CLASSIC, linewidth=2.5, label='Classic')
        ax.plot(models, m_q, marker='o', markersize=9, color=COLOR_QUANTUM, linewidth=2.5, label='Quantum')
        
        ax.set_title(title, fontsize=15, fontweight='bold', pad=25)
        ax.set_ylabel(metric, fontsize=13)
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(fontsize=12, loc='lower right')
        ax.tick_params(axis='x', rotation=35)
        
        # --- ĐƯỜNG KẺ NGANG GIÁ TRỊ CAO NHẤT ---
        max_c = m_c.max()
        max_q = m_q.max()
        if pd.isna(max_c) and pd.isna(max_q):
            return
            
        global_max = max(max_c if pd.notna(max_c) else -999, max_q if pd.notna(max_q) else -999)
        
        champions = []
        for model in models:
            if pd.notna(m_c.get(model)) and np.isclose(m_c[model], global_max, atol=1e-5):
                champions.append(f"Classic {model}")
            if pd.notna(m_q.get(model)) and np.isclose(m_q[model], global_max, atol=1e-5):
                champions.append(f"Quantum {model}")
                
        champ_text = ", ".join(champions)
        
        ax.axhline(global_max, color='red', linestyle='-.', linewidth=1.5, alpha=0.7)
        ax.text(0, global_max + (global_max * 0.001), f" Tốt nhất: {global_max:.4f} ({champ_text})", 
                color='red', va='bottom', ha='left', fontsize=11, fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))
        
        # Tự động scale trục Y đẹp mắt
        min_val = min(m_c.min(), m_q.min())
        max_val = global_max
        margin = (max_val - min_val) * 0.15 if (max_val - min_val) > 0 else 0.05
        if pd.notna(min_val) and pd.notna(max_val):
            ax.set_ylim(min_val - margin, max_val + margin * 1.5)

    # Chart 1: AUC
    plot_line(ax1, c_f1, q_f1, 'ROC AUC', '1. Khả năng Phân tách Lõi (ROC AUC)')
    
    # Chart 2: Max F1
    plot_line(ax2, c_f1, q_f1, 'F1 Score', '2. Sức mạnh Tổng hợp (Max F1 Score)')
    
    # Chart 3: Recall khi Prec >= 0.9
    plot_line(ax3, c_rec_at_prec90, q_rec_at_prec90, 'Recall', '3. So sánh Recall khi bị ép Precision ≥ 0.90')
    
    # Chart 4: Precision khi Recall >= 0.9
    plot_line(ax4, c_prec_at_rec90, q_prec_at_rec90, 'Precision', '4. So sánh Precision khi bị ép Recall ≥ 0.90')
    
    # Chart 5: Precision khi Recall >= 0.85
    plot_line(ax5, c_prec_at_rec85, q_prec_at_rec85, 'Precision', '5. So sánh Precision khi bị ép Recall ≥ 0.85')
    
    plt.tight_layout()
    plt.show()
else:
    print("Vui lòng đảm bảo cả 4 file Excel đang nằm chung thư mục với Notebook!")
